# Multi-Agent Orchestration with AG2 and Mistral

[AG2](https://ag2.ai/) (formerly AutoGen) is an open-source multi-agent framework
with 400K+ monthly PyPI downloads. This notebook demonstrates how to build
multi-agent systems using AG2 with Mistral models.

We'll cover three patterns:
1. **Basic conversation** — two-agent chat with Mistral
2. **Function calling** — tool registration and execution
3. **GroupChat** — multi-agent orchestration with automatic speaker selection

*Author: [Faridun Mirzoev](https://github.com/faridun-ag2) — AG2 Team*

In [ ]:
!pip install -q ag2[openai]==0.11.4 mistralai==2.1.3 python-dotenv==1.0.1

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify API key is set
assert os.getenv("MISTRAL_API_KEY"), "Set MISTRAL_API_KEY environment variable"

## 1. Basic Two-Agent Conversation

AG2's fundamental pattern: an `AssistantAgent` (LLM-powered) talks to a
`UserProxyAgent` (represents the user, can execute tools).

AG2 supports Mistral natively via `MistralLLMConfigEntry` — no wrapper needed.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent, LLMConfig
from autogen.oai.mistral import MistralLLMConfigEntry

# AG2 native Mistral support
llm_config = LLMConfig(
    MistralLLMConfigEntry(
        model="mistral-large-latest",
        api_key=os.getenv("MISTRAL_API_KEY"),
    )
)

In [ ]:
assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful AI assistant powered by Mistral. "
        "Provide clear, concise answers."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
    code_execution_config=False,
)

In [ ]:
result = user_proxy.run(
    assistant,
    message="What makes Mistral models unique compared to other LLM providers?",
)
result.process()

## 2. Function Calling with Mistral

AG2 uses a dual registration pattern for tools:
- `register_for_llm()` — tells Mistral the tool exists (generates the JSON schema)
- `register_for_execution()` — tells the UserProxy how to execute it

Mistral Large supports function calling natively, and AG2 handles the schema
generation automatically from Python type hints.

In [ ]:
from typing import Annotated


def get_weather(
    city: Annotated[str, "City name to get weather for"],
) -> str:
    """Get the current weather for a city."""
    # Simulated weather data for demo
    weather_data = {
        "paris": "Paris: 18\u00b0C, partly cloudy, humidity 65%",
        "london": "London: 14\u00b0C, rainy, humidity 80%",
        "tokyo": "Tokyo: 22\u00b0C, sunny, humidity 55%",
        "new york": "New York: 20\u00b0C, clear skies, humidity 50%",
        "san francisco": "San Francisco: 16\u00b0C, foggy, humidity 75%",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")


def get_local_time(
    city: Annotated[str, "City name to get local time for"],
) -> str:
    """Get the current local time for a city."""
    # Simulated time data for demo
    times = {
        "paris": "14:30 CET (UTC+1)",
        "london": "13:30 GMT (UTC+0)",
        "tokyo": "22:30 JST (UTC+9)",
        "new york": "08:30 EST (UTC-5)",
        "san francisco": "05:30 PST (UTC-8)",
    }
    return times.get(city.lower(), f"Time data not available for {city}")

In [ ]:
travel_agent = AssistantAgent(
    name="TravelAgent",
    system_message=(
        "You are a travel assistant. Use the available tools to help "
        "users plan their trips. Always check weather and local time."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=5,
    code_execution_config=False,
)

# Register tools — AG2 dual pattern
travel_agent.register_for_llm(description="Get current weather")(get_weather)
travel_agent.register_for_llm(description="Get local time")(get_local_time)
user_proxy.register_for_execution()(get_weather)
user_proxy.register_for_execution()(get_local_time)

In [ ]:
result = user_proxy.run(
    travel_agent,
    message="I'm planning a trip to Paris and Tokyo next week. "
            "What's the weather like and what time is it there now?",
)
result.process()

## 3. Multi-Agent GroupChat

AG2's flagship feature: **GroupChat** with automatic speaker selection.
Define specialist agents, and the `GroupChatManager` uses Mistral to decide
which agent should respond next — no routing code needed.

In [ ]:
from autogen import GroupChat, GroupChatManager

researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research analyst. Gather facts and provide "
        "well-sourced analysis on the topic at hand."
    ),
    llm_config=llm_config,
)
writer = AssistantAgent(
    name="Writer",
    system_message=(
        "You are a skilled writer. Create clear, engaging content "
        "from research findings. Focus on readability."
    ),
    llm_config=llm_config,
)
critic = AssistantAgent(
    name="Critic",
    system_message=(
        "You review content for accuracy, clarity, and completeness. "
        "Provide constructive feedback. Say TERMINATE when satisfied."
    ),
    llm_config=llm_config,
)
user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

In [ ]:
groupchat = GroupChat(
    agents=[user_proxy, researcher, writer, critic],
    messages=[],
    max_round=10,
    speaker_selection_method="auto",  # Mistral selects the next speaker
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

In [ ]:
result = user_proxy.run(
    manager,
    message=(
        "Write a brief overview of the current state of open-source AI models "
        "in Europe, with a focus on Mistral AI's contributions."
    ),
)
result.process()

## Summary

| Pattern | AG2 Components | Use Case |
|---------|---------------|----------|
| Two-agent | `AssistantAgent` + `UserProxyAgent` | Simple Q&A, single tasks |
| Function calling | `register_for_llm` + `register_for_execution` | Tool use, API calls |
| Multi-agent | `GroupChat` + `GroupChatManager` | Complex workflows with specialists |

### Key Takeaways

- AG2 supports Mistral natively via `MistralLLMConfigEntry`
- Function calling works out of the box with Mistral Large
- GroupChat provides automatic multi-agent orchestration
- No wrapper libraries needed — just `pip install ag2[openai] mistralai`

### Resources

- [AG2 Documentation](https://docs.ag2.ai/)
- [AG2 GitHub](https://github.com/ag2ai/ag2)
- [Mistral AI API Reference](https://docs.mistral.ai/)